In [1]:
import matplotlib.pyplot as plt
import matplotlib
import PyQt5


# Use matplotlib Qt5Agg backend - best choice for MNE-Python interactive plotting functions
matplotlib.use('Qt5Agg')

import yaml
from data.eeg_loader import read_file, load_eeg, get_eeg_timestamps, load_stimulus, load_event, create_trial_events
config_file_path = '/Users/khanhha/eeg-auditory-stimulus/configs/claassen_cfg.yml'  # Replace with the actual path to your config file
with open(config_file_path, 'r') as file:
    config = yaml.safe_load(file)

eeg_path = r"/Users/khanhha/EEG_DATA/X~ X_ef7f7805-3781-4f1e-8134-8d5b57750330.EDF"
event_full_path = r"/Users/khanhha/EEG_DATA/patient_df.csv"

In [2]:
fname, raw = load_eeg(eeg_path, config)

Extracting EDF parameters from /Users/khanhha/EEG_DATA/X~ X_ef7f7805-3781-4f1e-8134-8d5b57750330.EDF...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2057215  =      0.000 ...  4017.998 secs...
Resampling data to 512 Hz
Sampling frequency of the instance is already 512.0, returning unmodified.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1691 samples (3.303 s)



/Users/khanhha/eeg-auditory-stimulus/data/eeg_loader.py:12: RuntimeWarning: Omitted 5 annotation(s) that were outside data range.
  raw = mne.io.read_raw_edf(eeg_path, preload=True)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.4s


In [3]:
import mne
raw2 = mne.io.read_raw_edf(eeg_path, preload=True)
raw2.describe()

Extracting EDF parameters from /Users/khanhha/EEG_DATA/X~ X_ef7f7805-3781-4f1e-8134-8d5b57750330.EDF...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2057215  =      0.000 ...  4017.998 secs...


/var/folders/t4/rtjrk9ds4gx0wybhjwbjhklr0000gn/T/ipykernel_80984/429113514.py:2: RuntimeWarning: Omitted 5 annotation(s) that were outside data range.
  raw2 = mne.io.read_raw_edf(eeg_path, preload=True)


<RawEDF | X~ X_ef7f7805-3781-4f1e-8134-8d5b57750330.EDF, 50 x 2057216 (4018.0 s), ~784.8 MB, data loaded>
ch  name   type  unit        min         Q1     median         Q3        max
 0  C3     EEG   µV      -285.26       9.06      18.95      28.84     585.53
 1  C4     EEG   µV      -254.94       2.80      13.35      23.90     521.25
 2  O1     EEG   µV      -761.53     -15.99      19.94      58.50    4665.91
 3  O2     EEG   µV      -293.17      -6.10      18.62      44.00     390.73
 4  FT9    EEG   µV      -436.22       5.11      32.47      60.48     391.72
 5  FT10   EEG   µV    -10800.00     -15.33      17.30      49.93   10800.00
 6  Cz     EEG   µV       -79.60      12.69      20.60      28.18      95.09
 7  F3     EEG   µV      -222.31       7.75      20.27      33.45     324.16
 8  F4     EEG   µV      -237.47      -2.80      10.71      25.87     321.19
 9  F7     EEG   µV      -892.05     -15.33      17.30      52.24     436.22
10  F8     EEG   µV     -1291.52     -20.93    

In [4]:
raw2.info

<Info | 8 non-empty values
 bads: []
 ch_names: C3, C4, O1, O2, FT9, FT10, Cz, F3, F4, F7, F8, Fz, Fp1, Fp2, ...
 chs: 50 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 256.0 Hz
 meas_date: 2024-09-17 09:58:57 UTC
 nchan: 50
 projs: []
 sfreq: 512.0 Hz
 subject_info: 4 items (dict)
>

In [5]:
raw2.plot()


Using matplotlib as 2D backend.


<MNEBrowseFigure size 2940x1780 with 4 Axes>

# Claassen Replicate

### Loading dependencies

In [7]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
from tqdm import tqdm_notebook

from sklearn.svm import LinearSVC, SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, cross_val_score, LeaveOneGroupOut

import mne
from mne.time_frequency import psd_array_multitaper
from mne.decoding import LinearModel, get_coef

## 1. Preprocessing Raw Data

In [24]:
# eeg_path

# # Read raw data
# raw = mne.io.read_raw_edf(eeg_path, preload=True)

# # Band-pass filter between 1-30 Hz over the readings
# fmin, fmax = 1, 30 # Hz
# raw.filter(l_freq=fmin, h_freq=fmax)

## 2. Reading events and segmenting trials into epochs

In [25]:
raw.info

<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, Fp2, Fz, F3, F4, F7, F8, Cz, C3, C4, T3, T4, Pz, P3, P4, ...
 chs: 19 EEG
 custom_ref_applied: False
 highpass: 1.0 Hz
 lowpass: 30.0 Hz
 meas_date: 2024-09-17 15:30:17 UTC
 nchan: 19
 projs: []
 sfreq: 512.0 Hz
 subject_info: 4 items (dict)
>

## 3. Get Events

Event array example:
events = array([[   1437,       0,       2],  
       [   1441,       0,       5],  
       [ 520408,       0,       3],  
       [1659458,       0,       3],  
       [1757552,       0,       3],  
       [1787115,       0,       3],  
       [2188080,       0,       1],  
       [3086259,       0,       3],  
       [3146126,       0,       3],  
       [3259333,       0,       3]])  

event_dict = {'10 HZ': 1,  
 'Clip Note': 2,  
 'XLSpike': 3,  
 'ending experiment 2': 4,  
 'starting experiment 2': 5}  

In [8]:
fname, raw = load_eeg(eeg_path, config)
start_time, end_time = get_eeg_timestamps(raw)
patient_id = load_stimulus(event_full_path, start_time, end_time)
df = pd.read_csv(event_full_path)
df = df.drop(columns=['Unnamed: 0'])
df = df[df['patient_id'] == patient_id]
df['start_time'] = pd.to_datetime(df['start_time'], unit='s', utc=True)
df['end_time'] = pd.to_datetime(df['end_time'], unit='s', utc=True)

Extracting EDF parameters from /Users/khanhha/EEG_DATA/X~ X_ef7f7805-3781-4f1e-8134-8d5b57750330.EDF...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2057215  =      0.000 ...  4017.998 secs...
Resampling data to 512 Hz
Sampling frequency of the instance is already 512.0, returning unmodified.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1691 samples (3.303 s)



/Users/khanhha/eeg-auditory-stimulus/data/eeg_loader.py:12: RuntimeWarning: Omitted 5 annotation(s) that were outside data range.
  raw = mne.io.read_raw_edf(eeg_path, preload=True)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.4s


In [10]:
events, event_dict = create_trial_events(df, start_time, config)
events, event_dict

(array([[ 403968,       0,       1],
        [ 413184,       0,       1],
        [ 421888,       0,       1],
        [ 430592,       0,       1],
        [ 439296,       0,       1],
        [ 448512,       0,       1],
        [ 457216,       0,       1],
        [ 465920,       0,       1],
        [ 475136,       0,       1],
        [ 483840,       0,       1],
        [ 492544,       0,       1],
        [ 501760,       0,       1],
        [ 510464,       0,       1],
        [ 519168,       0,       1],
        [ 527872,       0,       1],
        [ 537088,       0,       1],
        [ 545792,       0,       4],
        [ 562176,       0,       1],
        [ 571392,       0,       1],
        [ 580096,       0,       1],
        [ 589312,       0,       1],
        [ 598016,       0,       1],
        [ 606720,       0,       1],
        [ 615424,       0,       4],
        [ 631808,       0,       1],
        [ 640512,       0,       1],
        [ 649216,       0,       1],
 

In [29]:
# Create the metadata DataFrame
metadata = pd.DataFrame(dict(
    time_sample=events[:, 0],  # time sample in the events array
    id=events[:, 2],  # the unique code of the epoch (event code)
    move=(events[:, 2] % 2) == 1,  # whether the code corresponds to a 'move' trial
    instr=[key for key, value in event_dict.items() if value in events[:, 2]],  # mapping event codes to instructions
))

# Generate trial numbers based on event codes (assuming two instructions per trial)
metadata['trial'] = np.array(metadata['id']) // 2  # Change logic if necessary

ValueError: All arrays must be of the same length